# 🎙️ BhavVani Hindi SER — Training Pipeline
### Continues from your existing audio_vectors notebook
**Your existing code already does:** mp4→wav, librosa load, pitch, energy, 13 MFCCs → 15-dim feature vector

**This notebook does:** Extract & organize dataset → Build feature matrix → Train model → Evaluate → Save

---
## STEP 1 — Extract & Inspect the BhavVani ZIP Files

In [ ]:
import zipfile
import os

# ✅ Your 4 downloaded ZIP paths — update if needed
zip_files = [
    r"C:\Users\anish\Downloads\BhavVani-20260629T073441Z-3-001.zip",
    r"C:\Users\anish\Downloads\BhavVani-20260629T073441Z-3-002.zip",
    r"C:\Users\anish\Downloads\BhavVani-20260629T073441Z-3-003.zip",
    r"C:\Users\anish\Downloads\BhavVani-20260629T073441Z-3-004.zip",
]

# Extract all to a common folder
EXTRACT_DIR = r"C:\Users\anish\Downloads\BhavVani_Dataset"
os.makedirs(EXTRACT_DIR, exist_ok=True)

for zf in zip_files:
    print(f"Extracting {os.path.basename(zf)}...")
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(EXTRACT_DIR)

print("\n✅ All ZIPs extracted to:", EXTRACT_DIR)

In [ ]:
# Inspect what's inside — understand folder structure
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:  # Only show files 3 levels deep
        subindent = ' ' * 2 * (level + 1)
        for f in files[:5]:  # Show first 5 files per folder
            print(f"{subindent}{f}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files)-5} more files")

---
## STEP 2 — Find the Label CSV and Audio Files

In [ ]:
import pandas as pd

# Search for CSV label files in the extracted dataset
csv_files = []
audio_extensions = ('.wav', '.mp3', '.flac', '.ogg')
audio_count = 0

for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))
        if f.lower().endswith(audio_extensions):
            audio_count += 1

print(f"📄 CSV files found: {len(csv_files)}")
for c in csv_files:
    print(" ", c)

print(f"\n🎵 Total audio files found: {audio_count}")

In [ ]:
# Load and preview the label CSV
# BhavVani typically has a CSV with columns: filename, emotion/label

# Load the first CSV found (adjust index if needed)
label_df = pd.read_csv(csv_files[0])
print("Shape:", label_df.shape)
print("\nColumns:", label_df.columns.tolist())
print("\nFirst 5 rows:")
label_df.head()

In [ ]:
# Check emotion class distribution
# ⚠️ Replace 'emotion' with the actual column name from the output above
LABEL_COLUMN = 'emotion'   # 🔧 Change this if your CSV uses a different column name
FILE_COLUMN  = 'filename'  # 🔧 Change this if your CSV uses a different column name

print("Emotion class counts:")
print(label_df[LABEL_COLUMN].value_counts())

---
## STEP 3 — Feature Extraction Function
*(Same approach as your notebook — now applied to all 8734 files)*

In [ ]:
import librosa
import numpy as np
import imageio_ffmpeg as ffmpeg
import subprocess
import tempfile
import warnings
warnings.filterwarnings('ignore')

def convert_to_wav(input_path):
    """Convert any audio file to WAV 16kHz mono using ffmpeg (same as your notebook)"""
    if input_path.lower().endswith('.wav'):
        return input_path, False  # Already WAV, no temp file needed
    
    tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    tmp.close()
    subprocess.run([
        ffmpeg.get_ffmpeg_exe(), "-y",
        "-i", input_path,
        "-ac", "1", "-ar", "16000",
        tmp.name
    ], check=True, capture_output=True)
    return tmp.name, True  # Return temp file path + flag to delete later


def extract_features(file_path):
    """
    Extracts a feature vector from an audio file.
    Returns: numpy array of shape (40,)
      - 13 MFCC means
      - 13 MFCC std devs  
      - 12 Chroma features
      - 1  Pitch (mean)
      - 1  Energy
    """
    tmp_path, is_temp = convert_to_wav(file_path)
    
    try:
        audio, sr = librosa.load(tmp_path, sr=16000)
        
        if len(audio) == 0:
            return None
        
        # MFCCs — 13 mean + 13 std = 26 features
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfccs, axis=1)  # 13 values
        mfcc_std  = np.std(mfccs, axis=1)   # 13 values
        
        # Chroma — 12 features
        chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
        chroma_mean = np.mean(chroma, axis=1)  # 12 values
        
        # Pitch — 1 feature (same as your notebook)
        f0, _, _ = librosa.pyin(audio, fmin=75, fmax=600)
        pitch = np.array([np.nanmean(f0) if not np.all(np.isnan(f0)) else 0.0])
        
        # Energy — 1 feature (same as your notebook)
        energy = np.array([np.mean(audio ** 2)])
        
        # Combine → 40-dim vector
        feature_vector = np.concatenate([mfcc_mean, mfcc_std, chroma_mean, pitch, energy])
        return feature_vector
    
    except Exception as e:
        return None
    
    finally:
        if is_temp:
            os.unlink(tmp_path)


print("✅ Feature extraction function ready!")
print("   Feature vector size: 40 (13 MFCC mean + 13 MFCC std + 12 Chroma + 1 Pitch + 1 Energy)")

---
## STEP 4 — Build Feature Matrix from Entire Dataset
*(This will take 10–30 minutes for ~8734 files — run once and save)*

In [ ]:
from tqdm import tqdm

# ⚠️ IMPORTANT: Set the correct audio folder path here
# BhavVani audio files are typically in train/ test/ val/ subfolders
# Adjust this path based on what you saw in STEP 1 inspection
AUDIO_BASE_DIR = EXTRACT_DIR  # or a subfolder like os.path.join(EXTRACT_DIR, 'audio')

X = []  # Feature vectors
y = []  # Labels
failed = []

for idx, row in tqdm(label_df.iterrows(), total=len(label_df), desc="Extracting features"):
    
    # Build full audio file path
    # 🔧 Adjust based on how filename is stored in the CSV
    audio_path = os.path.join(AUDIO_BASE_DIR, row[FILE_COLUMN])
    
    if not os.path.exists(audio_path):
        failed.append(audio_path)
        continue
    
    features = extract_features(audio_path)
    
    if features is not None:
        X.append(features)
        y.append(row[LABEL_COLUMN])
    else:
        failed.append(audio_path)

X = np.array(X)
y = np.array(y)

print(f"\n✅ Features extracted!")
print(f"   X shape: {X.shape}  (samples × features)")
print(f"   y shape: {y.shape}")
print(f"   Failed/missing: {len(failed)}")

In [ ]:
# 💾 Save the extracted features so you don't have to re-run the loop
SAVE_DIR = r"C:\Users\anish\Downloads\BhavVani_Features"
os.makedirs(SAVE_DIR, exist_ok=True)

np.save(os.path.join(SAVE_DIR, 'X_features.npy'), X)
np.save(os.path.join(SAVE_DIR, 'y_labels.npy'), y)

print(f"✅ Saved to {SAVE_DIR}")
print("   X_features.npy  →", X.shape)
print("   y_labels.npy    →", y.shape)

In [ ]:
# ⚡ Next time, LOAD directly instead of re-extracting:
# X = np.load(os.path.join(SAVE_DIR, 'X_features.npy'))
# y = np.load(os.path.join(SAVE_DIR, 'y_labels.npy'), allow_pickle=True)

---
## STEP 5 — Encode Labels & Split Data

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Encode string emotion labels → numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Emotion classes:", le.classes_)
print("Encoded as:     ", list(range(len(le.classes_))))

# Normalize features (important for most ML models)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train / Val / Test split  (70% / 15% / 15%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nTrain: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

---
## STEP 6A — Train: SVM (Fast Baseline, Great Accuracy)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

print("Training SVM...")
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
svm_model.fit(X_train, y_train)

# Evaluate
y_pred_svm = svm_model.predict(X_test)
acc = accuracy_score(y_test, y_pred_svm)
print(f"\n✅ SVM Test Accuracy: {acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('SVM Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

---
## STEP 6B — Train: MLP Neural Network (Better Accuracy)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Convert to tensors
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_v  = torch.tensor(X_val,   dtype=torch.float32)
y_v  = torch.tensor(y_val,   dtype=torch.long)
X_te = torch.tensor(X_test,  dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_v,  y_v),  batch_size=64)

NUM_CLASSES = len(le.classes_)
INPUT_SIZE  = X_train.shape[1]  # 40

print(f"Input features: {INPUT_SIZE}")
print(f"Number of classes: {NUM_CLASSES} → {list(le.classes_)}")

In [ ]:
# MLP Model
class EmotionMLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = EmotionMLP(INPUT_SIZE, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(model)

In [ ]:
# Training Loop
EPOCHS = 50
best_val_acc = 0
train_losses, val_accs = [], []

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # --- Validate ---
    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
    
    val_acc = correct / len(X_val)
    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)
    
    train_losses.append(avg_loss)
    val_accs.append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_emotion_model.pt'))
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Loss: {avg_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Best: {best_val_acc*100:.2f}%")

print(f"\n✅ Training complete! Best Val Accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot([v*100 for v in val_accs])
ax2.set_title('Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True)

plt.tight_layout()
plt.show()

---
## STEP 7 — Evaluate Best Model on Test Set

In [ ]:
# Load best saved model
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'best_emotion_model.pt')))
model.eval()

with torch.no_grad():
    logits = model(X_te.to(device))
    y_pred_mlp = logits.argmax(dim=1).cpu().numpy()

test_acc = accuracy_score(y_test, y_pred_mlp)
print(f"✅ MLP Test Accuracy: {test_acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp, target_names=le.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_mlp)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('MLP Confusion Matrix — Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

---
## STEP 8 — Save Everything for Future Use

In [ ]:
import joblib

# Save label encoder
joblib.dump(le, os.path.join(SAVE_DIR, 'label_encoder.pkl'))

# Save scaler
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler.pkl'))

# Save SVM model
joblib.dump(svm_model, os.path.join(SAVE_DIR, 'svm_model.pkl'))

# MLP model already saved as best_emotion_model.pt

print("✅ Saved:")
print(f"  {SAVE_DIR}/label_encoder.pkl")
print(f"  {SAVE_DIR}/scaler.pkl")
print(f"  {SAVE_DIR}/svm_model.pkl")
print(f"  {SAVE_DIR}/best_emotion_model.pt")

---
## STEP 9 — 🎤 Predict Emotion on Your Own Voice
*(Uses your existing audio vector pipeline!)*

In [ ]:
def predict_emotion(audio_file_path, use_model='mlp'):
    """
    Predict emotion from an audio file.
    use_model: 'mlp' or 'svm'
    """
    # Extract features
    features = extract_features(audio_file_path)
    if features is None:
        print("❌ Could not extract features from file")
        return
    
    # Scale
    features_scaled = scaler.transform([features])
    
    if use_model == 'svm':
        pred = svm_model.predict(features_scaled)[0]
        proba = svm_model.predict_proba(features_scaled)[0]
    else:
        with torch.no_grad():
            inp = torch.tensor(features_scaled, dtype=torch.float32).to(device)
            logits = model(inp)
            proba = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred = proba.argmax()
    
    emotion = le.inverse_transform([pred])[0]
    confidence = proba.max() * 100
    
    print(f"\n🎯 Predicted Emotion: {emotion.upper()}  ({confidence:.1f}% confident)")
    print("\nAll probabilities:")
    for em, prob in sorted(zip(le.classes_, proba), key=lambda x: -x[1]):
        bar = '█' * int(prob * 20)
        print(f"  {em:12s} {bar:20s} {prob*100:.1f}%")
    
    return emotion


# ✅ Test with your existing voice.wav from your notebook!
predict_emotion("voice.wav", use_model='mlp')

---
## 📝 Troubleshooting Notes

| Issue | Fix |
|---|---|
| Audio path not found | Check `FILE_COLUMN` name and path prefix in Step 4 |
| Wrong column names | Print `label_df.columns` and update `LABEL_COLUMN` / `FILE_COLUMN` |
| Low accuracy (<60%) | Try adding ZCR, spectral centroid, or mel-spectrogram features |
| CUDA out of memory | Reduce batch_size from 64 to 32 |
| tqdm not installed | Run: `pip install tqdm` |
| joblib not installed | Run: `pip install joblib` |